<a href="https://colab.research.google.com/github/memetciftsuren/Dioxamine/blob/main/tek_resimden_3d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1) Ekran kartı var mı?
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Ekran kartı yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet, sonra yeniden çalıştır'
BF16_OK = torch.cuda.get_device_capability()[0] >= 8   # T4 = 7.5 → bf16 yok, kodlayıcılar fp32'de çalışır
print('✓', torch.cuda.get_device_name(), '| torch', torch.__version__)

# Tek resimden 3D — ücretsiz Colab defteri

Bir resmi, etrafında dönebileceğin bir **3D Gaussian splat** nesnesine çevirir. Model: Speridlabs **Idol** (açık kaynak, Apache 2.0).
Ağır işi Google'ın ücretsiz **T4** ekran kartı yapar; bilgisayarında hiçbir şey kurman gerekmez. Kod yazman da gerekmez.

Hazırlayan: **memoLABS** (YouTube). Videoda her adım tek tek gösteriliyor.

---

### Başlamadan önce (bir kez)

**0. Kendi kopyanı al:** üst menü **Dosya → Drive'a kopya kaydet**. Bundan sonra o kopyada çalış.

**1. Ücretsiz ekran kartı:** **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**, sonra sağ üstten **Bağlan**.

**2. İki kilitli kapı (Hugging Face):** Idol, iki başka şirketin modelini kullanır. Para istemezler ama koşulları kabul etmeni isterler.
- https://huggingface.co adresinde ücretsiz hesap aç, e-postanı doğrula.
- Giriş yapmışken bu iki sayfada en üstteki formu doldur ve kabul et:
  - DINOv3 (Meta): https://huggingface.co/facebook/dinov3-vith16plus-pretrain-lvd1689m
  - FLUX.2-dev (Black Forest Labs, yalnız VAE parçası iner): https://huggingface.co/black-forest-labs/FLUX.2-dev
- Bazen onay elle verilir ve biraz sürer. Durum: https://huggingface.co/settings/gated-repos (ikisinde de *accepted* olmalı).

**3. Anahtar (token):** https://huggingface.co/settings/tokens → **Create new token** → tür **Read** → kopyala. Kimseyle paylaşma.
Colab'da soldaki **🔑 (Gizli anahtarlar)** → **Yeni gizli anahtar ekle** → ad `HF_TOKEN`, değer token, **Not defteri erişimi** açık.
(Bunu atlarsan 3. hücre token'ı bir kutuya yapıştırmanı ister.)

### Çalıştır
**Çalışma zamanı → Tümünü çalıştır.** 2. hücre birkaç dakika sürer; bu sırada sekmeyi yenileme.
4. hücrede **Dosya seç** düğmesi çıkar: resmini (ya da birkaç resmi) yükle. İlk seferde model ~3 GB indirir.
T4'te resim başına birkaç dakika (bizde 4–5 dk) sürer. Sonunda `idol_out.zip` iner.

### Zip'te ne var? (her resim için)
| Dosya | Ne |
|---|---|
| `<ad>.ply` | **3D dosyan** (Gaussian splat) |
| `<ad>_turntable.mp4` | nesnenin etrafında dönen video |
| `<ad>_sheet.png` | 12 açıdan görünüş |
| `<ad>_check.png` | kontrol: solda modelin senin açından gördüğü, sağda senin resmin. Benziyorsa model resmini doğru anlamış |

**`.ply` dosyanı aç:** https://superspl.at/editor → dosyayı sayfaya sürükle-bırak. Ücretsiz, hesap istemez; dosyan bilgisayarında kalır.
Fareyle döndür, tekerlekle yaklaş; uçuşan lekeleri seçip sil, HTML olarak dışa aktar.

### İyi sonuç için resim
- **Tek bir nesne** olsun.
- Nesne kadraja **tam sığsın**, kenardan kesilmesin.
- **Sade arka plan**; sıkıştırılmış JPEG yerine temiz bir PNG daha iyi. Arka planı model kendisi siler.
- Sert, basit şekilli nesneler (soba, kupa, kaykay, ürün) en iyisi. Karakterler önden iyi, arkası tahmin: model görmediği yeri uydurur.
- Bu bir leke bulutudur, üçgenli model (mesh) değil: oyun motoruna ya da 3D yazıcıya doğrudan gitmez.

### Takılırsan
- **"Bağlanıyor"da uzun süre bekliyorsa** ya da *Dosya seç* düğmesi çalışmıyorsa: hücreyi durdur → **Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil** → T4 ile yeniden bağlan → Tümünü çalıştır.
- **Hücrelerin başında kırmızı ünlem**: o hücreler hata vermedi, önceki bir hata yüzünden hiç çalışmadı. Hatayı düzelt, takıldığın hücreye tıkla → **Çalışma zamanı → Sonrakini çalıştır**.
- **3. hücre "kilitli" diyorsa**: o modelin sayfasında koşulları kabul etmemişsin ya da onay daha gelmemiş (yukarıdaki 2. adım).
- Ücretsiz Colab'ın kaynakları garanti değildir; bazen T4 verilmez. Biraz sonra yeniden dene.

### Lisans
Idol: Apache 2.0. Kodlayıcılar kendi lisanslarıyla iner: DINOv3 (Meta, DINOv3 License; dağıtımda "Built with DINOv3") ve
FLUX.2-dev VAE (Black Forest Labs, FLUX Non-Commercial License). Ticari bir işte kullanacaksan lisans metinlerini kendin oku.
Bu defter Idol SDK'yı (https://github.com/speridlabs/idol-sdk) olduğu gibi kullanır; T4'te çalışsın diye yalnız sayı biçimini (bf16 → fp32) değiştirir.

In [ ]:
# 2) Kurulum: Idol SDK + gsplat (splat çizimi). Birkaç dakika sürebilir; sekmeyi yenileme.
%cd /content
!rm -rf idol-sdk && git clone -q --depth 1 https://github.com/speridlabs/idol-sdk
if not BF16_OK:
    # T4: SDK, FLUX.2 VAE kodlayıcısını bf16'ya sabitliyor → fp32 (sonuç aynı, yalnız biraz daha yavaş)
    p = '/content/idol-sdk/idol/conditioning.py'
    s = open(p).read().replace('torch_dtype=torch.bfloat16', 'torch_dtype=torch.float32').replace('images.to(torch.bfloat16)', 'images.to(torch.float32)')
    open(p, 'w').write(s)
!pip -q install -e /content/idol-sdk
!pip -q install gsplat ninja imageio imageio-ffmpeg
import sys
if '/content/idol-sdk' not in sys.path: sys.path.insert(0, '/content/idol-sdk')   # "No module named 'idol'" hatasını önler
print('✓ kurulum tamam')

In [ ]:
# 3) Hugging Face girişi + iki kilitli modelin erişim kontrolü
import os
from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = None
if not tok:
    from getpass import getpass
    tok = getpass('Hugging Face token (Read) yapıştır, Enter: ').strip()
login(token=tok)

GATED = {'facebook/dinov3-vith16plus-pretrain-lvd1689m': ('DINOv3 (Meta)', 'config.json'),
         'black-forest-labs/FLUX.2-dev': ('FLUX.2-dev (Black Forest Labs)', 'vae/config.json')}
missing = []
for repo, (label, probe) in GATED.items():
    try:
        hf_hub_download(repo, probe, token=tok)
        print('✓ erişim var:', label)
    except Exception as e:
        missing.append(f'  ✗ {label}: https://huggingface.co/{repo}  ({type(e).__name__})')
if missing:
    raise SystemExit('Kilit açık değil. Bu sayfalarda giriş yapmışken formu doldurup kabul et; onay bekliyorsa '
                     'https://huggingface.co/settings/gated-repos adresinden takip et:\n' + '\n'.join(missing))

In [ ]:
# 4) Resmini yükle: "Dosya seç" → bir ya da birkaç resim (PNG önerilir; JPG de olur)
import re
from google.colab import files
os.makedirs('/content/in', exist_ok=True); os.chdir('/content/in')
up = files.upload()
IMAGES = []
for n in up:
    safe = re.sub(r'[^A-Za-z0-9._-]+', '_', n)          # boşluklu / Türkçe karakterli adlar sorun çıkarmasın
    if safe != n: os.replace(n, safe)
    IMAGES.append(f'/content/in/{safe}')
IMAGES.sort()
assert IMAGES, 'Resim seçilmedi: bu hücreyi yeniden çalıştır ve Dosya seç ile resmini yükle'
print('✓', IMAGES)

In [ ]:
# 5) Ayarlar (dokunmadan da çalışır)
MODE = 'synthetic'        # çizim / render / yapay zekâ görseli: 'synthetic' · gerçek fotoğraf: 'real'
SEED = 42                 # farklı bir sonuç denemek için değiştir
STEPS = 50                # Idol varsayılanı; hızlı ön deneme için 20
NUM_GAUSSIANS = 262144    # leke sayısı (32768 – 1048576)
MIN_OPACITY = 0.05        # neredeyse saydam lekeleri at (pus)
FRAMES = 72               # dönüş videosu: 72 kare = 5° adım
SIZE = 768                # dönüş karesi (px)
EXTRA_PITCH_DEG = 0.0     # dönüşte kamerayı yukarı (+) / aşağı (−) eğ
BG = (32, 32, 36)         # video ve sayfa arka planı
SAVE_FRAMES = False       # True: dönüş karelerini saydam PNG olarak da zip'e koy

In [ ]:
# 6) Idol modelini yükle (ilk seferde ~3 GB iner)
from idol import IdolPipeline
pipe = IdolPipeline.from_pretrained('speridlabs/idol', dinov3_dtype='bf16' if BF16_OK else 'fp32', optimize=BF16_OK)
print('✓ model hazır')

In [ ]:
# 7) Çizim yardımcıları (gsplat). Idol'un iç uzayı z-yukarı; giriş kamerasını kendisi tahmin eder (yaw, pitch, radius, fov).
import math, json, numpy as np, torch
from PIL import Image
from gsplat import rasterization

def look_at(pos, target=(0, 0, 0), up=(0, 0, 1)):
    p, t, u = (torch.tensor(v, dtype=torch.float32) for v in (pos, target, up))
    f = torch.nn.functional.normalize(t - p, dim=0)
    r = torch.nn.functional.normalize(torch.linalg.cross(f, u), dim=0)
    d = torch.linalg.cross(f, r)
    R = torch.stack([r, d, f])                         # OpenCV: x sağ, y aşağı, z ileri
    M = torch.eye(4); M[:3, :3] = R; M[:3, 3] = -R @ p
    return M

def cam_pos(yaw, pitch, radius, sign):
    return [sign * radius * math.cos(pitch) * math.sin(yaw), sign * radius * math.cos(pitch) * math.cos(yaw), sign * radius * math.sin(pitch)]

def splat_tensors(g, min_opacity):
    op = g.opacity.reshape(-1).float()
    keep = op >= min_opacity
    return dict(means=g.xyz[keep].float().contiguous(), quats=g.rotation[keep].float().contiguous(),
                scales=g.scaling[keep].float().contiguous(), opacities=op[keep].contiguous(),
                colors=g.features_dc[keep].float().contiguous())

def render(S, pos, fov_deg, size):
    vm = look_at(pos).cuda()[None]
    f = 0.5 * size / math.tan(math.radians(fov_deg) / 2)
    K = torch.tensor([[f, 0, size / 2], [0, f, size / 2], [0, 0, 1]], dtype=torch.float32, device='cuda')[None]
    rgb, alpha, _ = rasterization(S['means'], S['quats'], S['scales'], S['opacities'], S['colors'], vm, K, size, size,
                                  sh_degree=0, near_plane=0.01, far_plane=1e4)
    a = alpha[0, ..., 0].clamp(0, 1)
    c = (rgb[0] / a[..., None].clamp_min(1e-6)).clamp(0, 1)          # önceden çarpılmış → düz RGBA
    out = torch.cat([c, a[..., None]], -1).mul(255).round().byte().cpu().numpy()
    return Image.fromarray(out, 'RGBA')

def iou(a, b):
    a, b = np.asarray(a) > 127, np.asarray(b) > 127
    return (a & b).sum() / max(1, (a | b).sum())

def fit_to(im, ref):
    """render'ı (RGBA) senin resmindeki nesnenin kutusuna oturt (ölçek + kaydırma): boyut farkı karşılaştırmayı bozmasın"""
    A = np.asarray(im.getchannel('A')) > 127; M = np.asarray(ref).max(-1) > 8
    if A.sum() < 10 or M.sum() < 10: return im
    ay, ax = np.where(A); my, mx = np.where(M)
    crop = im.crop((ax.min(), ay.min(), ax.max() + 1, ay.max() + 1)).resize((mx.max() - mx.min() + 1, my.max() - my.min() + 1), Image.LANCZOS)
    out = Image.new('RGBA', ref.size, (0, 0, 0, 0)); out.paste(crop, (int(mx.min()), int(my.min()))); return out

def match(im, ref, plain=False):
    """siluet örtüşmesi (IoU) − renk farkı: önden mi arkadan mı bakıldığını renk ayırır"""
    a = np.asarray(im).astype(np.float32) / 255; r = np.asarray(ref).astype(np.float32) / 255
    A = a[..., 3] > 0.5; M = r.max(-1) > 8 / 255
    io = (A & M).sum() / max(1, (A | M).sum())
    if plain: return io
    both = A & M
    return io - 0.6 * (np.abs(a[..., :3][both] - r[both]).mean() if both.sum() else 1)

def on_bg(im, bg=BG):
    base = Image.new('RGBA', im.size, bg + (255,)); base.alpha_composite(im); return base.convert('RGB')
print('✓')

In [ ]:
# 8) Üret: her resim için .ply + kontrol resmi + 12 açılık sayfa + dönüş videosu (T4'te resim başına birkaç dakika)
import subprocess, shutil, time
OUT = '/content/out'; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
for path in IMAGES:
    name = os.path.splitext(os.path.basename(path))[0]
    t0 = time.time(); print('▶', name)
    res = pipe(path, mode=MODE, seed=SEED, steps=STEPS, num_gaussians=NUM_GAUSSIANS, verbose=True)
    res.gaussian.save_ply(f'{OUT}/{name}.ply')
    res.preprocessed_image.save(f'{OUT}/{name}_input.png')
    cam = res.camera; json.dump(cam, open(f'{OUT}/{name}_camera.json', 'w'), indent=2)
    S = splat_tensors(res.gaussian, MIN_OPACITY)
    print(f"  leke: {len(S['opacities'])}")

    # giriş açısı: render senin resminin kutusuna oturtulur, siluet + renk benzerliğiyle en iyi bakış seçilir
    # (Idol'un kamera tahmini işaret/eksen olarak farklı olabilir: yön, ayna, 24 yaw adımı ve pitch işareti taranır)
    inp = res.preprocessed_image.convert('RGB'); n = inp.size[0]
    small = inp.resize((160, 160))
    best = None
    for sign in (1, -1):
        for pz in (1, -1):
            for k in range(24):
                yaw = cam['yaw'] + 2 * math.pi * k / 24; pitch = pz * cam['pitch']
                im = render(S, cam_pos(yaw, pitch, cam['radius'], sign), cam['fov'], 160)
                for mirror in (False, True):
                    m = im.transpose(Image.FLIP_LEFT_RIGHT) if mirror else im
                    sc = match(fit_to(m, small), small)
                    if best is None or sc > best[0]: best = (sc, sign, mirror, yaw, pitch)
    _, SIGN, MIRROR, YAW0, PITCH0 = best
    chk = render(S, cam_pos(YAW0, PITCH0, cam['radius'], SIGN), cam['fov'], n)
    if MIRROR: chk = chk.transpose(Image.FLIP_LEFT_RIGHT)
    chk = fit_to(chk, inp)
    Image.fromarray(np.concatenate([np.asarray(on_bg(chk)), np.asarray(inp)], 1)).save(f'{OUT}/{name}_check.png')
    print(f'  kontrol benzerliği: {match(chk, inp, plain=True):.2f}  (1.00 = birebir; 0.80 üstü iyi)')

    td = f'/content/frames/{name}'; shutil.rmtree(td, ignore_errors=True); os.makedirs(td)
    pitch = PITCH0 + math.radians(EXTRA_PITCH_DEG)
    frames = []
    for i in range(FRAMES):
        yaw = YAW0 + 2 * math.pi * i / FRAMES
        im = render(S, cam_pos(yaw, pitch, cam['radius'], SIGN), cam['fov'], SIZE)
        if MIRROR: im = im.transpose(Image.FLIP_LEFT_RIGHT)
        im.save(f'{td}/{i:03d}.png'); frames.append(im)
    step = max(1, FRAMES // 12); pick = frames[::step][:12]; w = SIZE // 2
    sheet = Image.new('RGB', (w * 6, w * 2), BG)
    for k, im in enumerate(pick): sheet.paste(on_bg(im).resize((w, w)), ((k % 6) * w, (k // 6) * w))
    sheet.save(f'{OUT}/{name}_sheet.png')
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-framerate', '24', '-i', f'{td}/%03d.png', '-vf',
                    f'color=c=0x{BG[0]:02x}{BG[1]:02x}{BG[2]:02x}:s={SIZE}x{SIZE}[b];[b][0]overlay=shortest=1,format=yuv420p',
                    f'{OUT}/{name}_turntable.mp4'], check=True)
    if SAVE_FRAMES: shutil.copytree(td, f'{OUT}/{name}_turntable')
    del res, S; torch.cuda.empty_cache()
    print(f'  ✓ {name}: {time.time() - t0:.0f} sn')
print('✓ bitti:', sorted(os.listdir(OUT)))

In [ ]:
# 9) Sonuca bak
from IPython.display import display, Video
for path in IMAGES:
    name = os.path.splitext(os.path.basename(path))[0]
    print(name, '— kontrol (solda modelin gördüğü, sağda senin resmin):'); display(Image.open(f'{OUT}/{name}_check.png').resize((1024, 512)))
    print(name, '— 12 açı:'); display(Image.open(f'{OUT}/{name}_sheet.png'))
    display(Video(f'{OUT}/{name}_turntable.mp4', embed=True, width=480))

In [ ]:
# 10) İndir: idol_out.zip. İçindeki .ply dosyasını https://superspl.at/editor sayfasına sürükle-bırak.
!cd /content && rm -f idol_out.zip && cd out && zip -qr ../idol_out.zip .
files.download('/content/idol_out.zip')